# DTP Managed Roads — Silver Transformation

## Purpose

Transform the validated Bronze DTP Managed Roads dataset into a consistent,
analysis-ready road-segment dataset for Asset Intelligence reporting.

## Silver Objectives

- Standardise blank and whitespace-only values to NULL
- Preserve legitimate missingness without arbitrary imputation
- Identify and flag incomplete core road attributes
- Standardise reporting attributes
- Preserve record and identifier integrity
- Validate geospatial integrity
- Derive road-segment length using projected geometry
- Create an analysis-ready dataset for Gold aggregation and Power BI


In [0]:
%pip install geopandas pyogrio

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
%restart_python

## Imports

In [0]:
import geopandas as gpd
import pandas as pd
import numpy as np

print("Libraries loaded successfully")

Libraries loaded successfully


## Load Bronze Dataset

Load the validated Bronze output produced by the ingestion notebook.

In [0]:
bronze_path = "/Volumes/dtp_data/dtp_schema/dtp_geojson/managed_roads_bronze.geojson"

roads_bronze = gpd.read_file(bronze_path)

print("=== BRONZE INPUT ===")
print(f"Records : {len(roads_bronze):,}")
print(f"Columns : {len(roads_bronze.columns)}")
print(f"CRS     : {roads_bronze.crs}")


=== BRONZE INPUT ===
Records : 90,797
Columns : 24
CRS     : EPSG:4326


In [0]:
assert roads_bronze["OBJECTID"].isna().sum() == 0, \
    "Missing OBJECTIDs detected in Bronze"

assert roads_bronze["OBJECTID"].duplicated().sum() == 0, \
    "Duplicate OBJECTIDs detected in Bronze"

assert roads_bronze.geometry.isna().sum() == 0, \
    "Missing geometries detected in Bronze"

print("Bronze input validation: PASS")

Bronze input validation: PASS


## Create Silver Working Dataset

Create an independent working copy of the validated Bronze dataset before
applying Silver transformations.

In [0]:
roads_silver = roads_bronze.copy()

print("Silver working copy created")
print(f"Records: {len(roads_silver):,}")

Silver working copy created
Records: 90,797


## Text Value Standardisation

Standardise blank and whitespace-only text values to NULL so missingness
can be assessed consistently across the dataset.

In [0]:
text_cols = roads_silver.select_dtypes(include="object").columns.tolist()

for col in text_cols:
    roads_silver[col] = (
        roads_silver[col]
        .astype("string")
        .str.strip()
        .replace("", pd.NA)
    )

print("Blank and whitespace-only strings converted to NULL")

Blank and whitespace-only strings converted to NULL


In [0]:
remaining_blanks = []

for col in roads_silver.select_dtypes(
    include=["object", "string"]
).columns:

    values = roads_silver[col].dropna()

    blank_count = (
        values
        .astype(str)
        .str.strip()
        .eq("")
        .sum()
    )

    remaining_blanks.append({
        "column": col,
        "remaining_blank_strings": blank_count
    })

remaining_blank_profile = pd.DataFrame(remaining_blanks)

remaining_blank_issues = remaining_blank_profile[
    remaining_blank_profile["remaining_blank_strings"] > 0
]

if remaining_blank_issues.empty:
    print("Validation PASS: no blank or whitespace-only strings remain.")
else:
    display(remaining_blank_issues)

Validation PASS: no blank or whitespace-only strings remain.


## Post-Standardisation Missingness Assessment

Assess attribute completeness after blank strings have been standardised to
proper NULL values.

In [0]:
silver_missing = pd.DataFrame({
    "column": roads_silver.columns,
    "missing_count": roads_silver.isna().sum().values,
    "missing_pct": (
        roads_silver.isna().mean() * 100
    ).round(2).values
})

silver_missing = (
    silver_missing[
        silver_missing["missing_count"] > 0
    ]
    .sort_values(
        "missing_pct",
        ascending=False
    )
    .reset_index(drop=True)
)

display(silver_missing)

column,missing_count,missing_pct
ALT2_SUFFIX,90661,99.85
ALT_SUFFIX,90531,99.71
RD_SUFFIX,89870,98.98
LOCAL_SUFFIX,88264,97.21
ALT2_NAME,80656,88.83
ALT2_TYPE,80650,88.82
ALT_TYPE,68367,75.3
ALT_NAME,68333,75.26
DEC_TYPE,1321,1.45
RD_TYPE,1321,1.45


In [0]:
fully_empty_cols = [
    col
    for col in roads_silver.columns
    if roads_silver[col].isna().all()
]

print("Fully empty columns:")
print(fully_empty_cols)

Fully empty columns:
[]


## Core Attribute Completeness Review

Review incomplete values in the principal road-type attributes separately
from optional alternative-name and suffix fields.

Records are retained rather than deleted or automatically imputed.

In [0]:
core_missing = roads_silver[
    roads_silver[
        ["DEC_TYPE", "RD_TYPE", "LOCAL_TYPE"]
    ]
    .isna()
    .any(axis=1)
].copy()

print(
    "Records requiring core attribute review:",
    f"{len(core_missing):,}"
)

display(
    core_missing[
        [
            "OBJECTID",
            "DEC_NAME",
            "DEC_TYPE",
            "RD_NAME",
            "RD_TYPE",
            "LOCAL_NAME",
            "LOCAL_TYPE",
            "CLASSN",
            "RMACLASS",
            "LOCALITY"
        ]
    ].head(50)
)

Records requiring core attribute review: 1,763


OBJECTID,DEC_NAME,DEC_TYPE,RD_NAME,RD_TYPE,LOCAL_NAME,LOCAL_TYPE,CLASSN,RMACLASS,LOCALITY
970,WARBURTON-WOODS POINT,ROAD,WARBURTON-WOODS POINT,ROAD,UNNAMED,null,MR,AO,MATLOCK
971,WARBURTON-WOODS POINT,ROAD,WARBURTON-WOODS POINT,ROAD,UNNAMED,null,MR,AO,JERICHO
2490,PRINCES,HIGHWAY EAST,PRINCES,HIGHWAY EAST,UNNAMED,null,HW,AH,LAKES ENTRANCE
2562,PRINCES,HIGHWAY EAST,PRINCES,HIGHWAY EAST,ESPLANADE,null,HW,AH,LAKES ENTRANCE
2564,PRINCES,HIGHWAY EAST,PRINCES,HIGHWAY EAST,ESPLANADE,null,HW,AH,LAKES ENTRANCE
2565,PRINCES,HIGHWAY EAST,PRINCES,HIGHWAY EAST,ESPLANADE,null,HW,AH,LAKES ENTRANCE
2568,PRINCES,HIGHWAY EAST,PRINCES,HIGHWAY EAST,ESPLANADE,null,HW,AH,LAKES ENTRANCE
2571,PRINCES,HIGHWAY EAST,PRINCES,HIGHWAY EAST,ESPLANADE,null,HW,AH,LAKES ENTRANCE
2573,PRINCES,HIGHWAY EAST,PRINCES,HIGHWAY EAST,ESPLANADE,null,HW,AH,LAKES ENTRANCE
2575,PRINCES,HIGHWAY EAST,PRINCES,HIGHWAY EAST,ESPLANADE,null,HW,AH,LAKES ENTRANCE


## Core Attribute Data Quality Flags

Create transparent record-level indicators so downstream reporting can
identify road segments requiring attribute review without removing them
from network analysis.

In [0]:
roads_silver["DQ_MISSING_DEC_TYPE"] = (
    roads_silver["DEC_TYPE"].isna()
)

roads_silver["DQ_MISSING_RD_TYPE"] = (
    roads_silver["RD_TYPE"].isna()
)

roads_silver["DQ_MISSING_LOCAL_TYPE"] = (
    roads_silver["LOCAL_TYPE"].isna()
)

roads_silver["DQ_CORE_ATTRIBUTE_ISSUE"] = (
    roads_silver[
        ["DEC_TYPE", "RD_TYPE", "LOCAL_TYPE"]
    ]
    .isna()
    .any(axis=1)
)

roads_silver["DQ_ATTRIBUTE_STATUS"] = np.where(
    roads_silver["DQ_CORE_ATTRIBUTE_ISSUE"],
    "REVIEW",
    "COMPLETE"
)

print("Core attribute quality flags created")

Core attribute quality flags created


In [0]:
dq_attribute_summary = (
    roads_silver["DQ_ATTRIBUTE_STATUS"]
    .value_counts()
    .rename_axis("DQ_ATTRIBUTE_STATUS")
    .reset_index(name="record_count")
)

dq_attribute_summary["record_pct"] = (
    dq_attribute_summary["record_count"]
    / len(roads_silver)
    * 100
).round(2)

display(dq_attribute_summary)

DQ_ATTRIBUTE_STATUS,record_count,record_pct
COMPLETE,89034,98.06
REVIEW,1763,1.94


## Categorical Attribute Standardisation

Standardise selected coded and type attributes to consistent whitespace and
upper-case formatting.

Source codes are preserved; no descriptive meaning is inferred without an
authoritative business definition.

In [0]:
standardise_cols = [
    "DEC_TYPE",
    "RD_TYPE",
    "LOCAL_TYPE",
    "CLASSN",
    "RMACLASS",
    "SRNS",
    "RD_SECTION"
]

for col in standardise_cols:
    roads_silver[col] = (
        roads_silver[col]
        .astype("string")
        .str.strip()
        .str.upper()
    )

print("Selected categorical attributes standardised")

Selected categorical attributes standardised


## Categorical Domain Review

Review the observed values and frequency distributions of important reporting
attributes before creating downstream business aggregations.

In [0]:
categorical_cols = [
    "CLASSN",
    "RMACLASS",
    "PROFILE",
    "RD_TYPE",
    "DEC_TYPE",
    "LOCAL_TYPE"
]

for col in categorical_cols:

    print(f"\n=== {col} ===")

    summary = (
        roads_silver[col]
        .value_counts(dropna=False)
        .rename_axis(col)
        .reset_index(name="record_count")
    )

    summary["record_pct"] = (
        summary["record_count"]
        / len(roads_silver)
        * 100
    ).round(2)

    display(summary)


=== CLASSN ===


CLASSN,record_count,record_pct
MR,52584,57.91
HW,25696,28.3
FW,7405,8.16
TR,4350,4.79
FR,572,0.63
NR,177,0.19
PR,13,0.01



=== RMACLASS ===


RMACLASS,record_count,record_pct
AO,56178,61.87
AH,27032,29.77
FW,7397,8.15
NR,177,0.19
PR,13,0.01



=== PROFILE ===


...

RD_TYPE,record_count,record_pct
ROAD,54589,60.12
HIGHWAY,21606,23.8
FREEWAY,4532,4.99
HIGHWAY EAST,2301,2.53
STREET,2144,2.36
HIGHWAY WEST,1572,1.73
null,1321,1.45
FREEWAY EAST,579,0.64
FREEWAY WEST,534,0.59
DRIVE,409,0.45



=== DEC_TYPE ===


DEC_TYPE,record_count,record_pct
ROAD,54589,60.12
HIGHWAY,21606,23.8
FREEWAY,4532,4.99
HIGHWAY EAST,2301,2.53
STREET,2144,2.36
HIGHWAY WEST,1572,1.73
null,1321,1.45
FREEWAY EAST,579,0.64
FREEWAY WEST,534,0.59
DRIVE,409,0.45



=== LOCAL_TYPE ===


LOCAL_TYPE,record_count,record_pct
ROAD,52027,57.3
HIGHWAY,17589,19.37
STREET,10156,11.19
FREEWAY,3550,3.91
RAMP,2338,2.57
null,991,1.09
AVENUE,975,1.07
DRIVE,768,0.85
PARADE,679,0.75
WAY,612,0.67


In [0]:
print("SRNS unique values:",
      roads_silver["SRNS"].nunique(dropna=True))

display(
    roads_silver["SRNS"]
    .value_counts(dropna=False)
    .head(20)
    .rename_axis("SRNS")
    .reset_index(name="record_count")
)

SRNS unique values: 481


SRNS,record_count
N,24023
M1,2491
A1,2073
B400,1749
A300,1531
M31,1110
A79,984
B100,874
B160,808
M8,798


## Identifier and Record Integrity

Confirm that Silver transformations have preserved the Bronze record count
and OBJECTID uniqueness.

In [0]:
print("=== IDENTIFIER INTEGRITY CHECK ===")

print(f"Bronze records      : {len(roads_bronze):,}")
print(f"Silver records      : {len(roads_silver):,}")

print(
    "Missing OBJECTIDs   :",
    roads_silver["OBJECTID"].isna().sum()
)

print(
    "Duplicate OBJECTIDs :",
    roads_silver["OBJECTID"].duplicated().sum()
)

assert len(roads_silver) == len(roads_bronze), \
    "Silver record count differs from Bronze"

assert roads_silver["OBJECTID"].isna().sum() == 0, \
    "Missing OBJECTIDs detected"

assert roads_silver["OBJECTID"].duplicated().sum() == 0, \
    "Duplicate OBJECTIDs detected"

print("\nIdentifier integrity: PASS")

=== IDENTIFIER INTEGRITY CHECK ===
Bronze records      : 90,797
Silver records      : 90,797
Missing OBJECTIDs   : 0
Duplicate OBJECTIDs : 0

Identifier integrity: PASS


## Geospatial Validation

Revalidate geometry after attribute transformations and before deriving
spatial measures.


In [0]:
print("=== SILVER GEOSPATIAL VALIDATION ===")

print(f"CRS: {roads_silver.crs}")

print(
    "Missing geometries:",
    roads_silver.geometry.isna().sum()
)

print(
    "Empty geometries:",
    roads_silver.geometry.is_empty.sum()
)

print(
    "Invalid geometries:",
    (~roads_silver.geometry.is_valid).sum()
)

print("\nGeometry types:")

print(
    roads_silver.geometry
    .geom_type
    .value_counts()
)

=== SILVER GEOSPATIAL VALIDATION ===
CRS: EPSG:4326
Missing geometries: 0
Empty geometries: 0
Invalid geometries: 0

Geometry types:
LineString    90797
Name: count, dtype: int64


## Spatial Measurement Preparation

The source dataset uses EPSG:4326 geographic coordinates.

A projected working copy is therefore created before calculating road-segment
length so that distance is measured in metres rather than angular degrees.

The main Silver geometry remains in EPSG:4326 for mapping.

In [0]:
roads_metric = roads_silver.to_crs(
    epsg=3111
)

print("Mapping CRS     :", roads_silver.crs)
print("Measurement CRS :", roads_metric.crs)

Mapping CRS     : EPSG:4326
Measurement CRS : EPSG:3111


In [0]:
roads_metric["SEGMENT_LENGTH_M"] = (
    roads_metric.geometry.length
)

roads_metric["SEGMENT_LENGTH_KM"] = (
    roads_metric["SEGMENT_LENGTH_M"]
    / 1000
)

print("Road segment lengths calculated in projected coordinates")

Road segment lengths calculated in projected coordinates


In [0]:
roads_silver["SEGMENT_LENGTH_M"] = (
    roads_metric["SEGMENT_LENGTH_M"]
    .to_numpy()
)

roads_silver["SEGMENT_LENGTH_KM"] = (
    roads_metric["SEGMENT_LENGTH_KM"]
    .to_numpy()
)

print("Derived length measures added to Silver")
print("Silver mapping CRS remains:", roads_silver.crs)

Derived length measures added to Silver
Silver mapping CRS remains: EPSG:4326


In [0]:
display(
    roads_silver[
        [
            "OBJECTID",
            "RD_NAME",
            "RD_TYPE",
            "CLASSN",
            "RMACLASS",
            "LOCALITY",
            "SEGMENT_LENGTH_M",
            "SEGMENT_LENGTH_KM"
        ]
    ].head(20)
)

OBJECTID,RD_NAME,RD_TYPE,CLASSN,RMACLASS,LOCALITY,SEGMENT_LENGTH_M,SEGMENT_LENGTH_KM
1,MACKENZIE,ROAD,MR,AO,WEST MELBOURNE,34.740411898025485,0.034740411898025486
2,MACKENZIE,ROAD,MR,AO,WEST MELBOURNE,44.767868208077275,0.04476786820807727
3,MACKENZIE,ROAD,MR,AO,WEST MELBOURNE,52.7438736000815,0.0527438736000815
4,MACKENZIE,ROAD,MR,AO,WEST MELBOURNE,19.106245845077993,0.019106245845077995
5,MACKENZIE,ROAD,MR,AO,WEST MELBOURNE,10.693296337946041,0.01069329633794604
6,MACKENZIE,ROAD,MR,AO,WEST MELBOURNE,81.83874593055961,0.0818387459305596
7,MACKENZIE,ROAD,MR,AO,WEST MELBOURNE,55.555896340076586,0.055555896340076585
8,MACKENZIE,ROAD,MR,AO,WEST MELBOURNE,48.23820120495433,0.048238201204954326
9,MACKENZIE,ROAD,MR,AO,WEST MELBOURNE,58.30401234522537,0.058304012345225364
10,MACKENZIE,ROAD,MR,AO,WEST MELBOURNE,34.03908455013818,0.034039084550138175


In [0]:
display(
    roads_silver[
        "SEGMENT_LENGTH_KM"
    ]
    .describe(
        percentiles=[
            0.01,
            0.05,
            0.25,
            0.50,
            0.75,
            0.95,
            0.99
        ]
    )
)

count    90797.000000
mean         0.289629
std          0.432533
min          0.002659
1%           0.008546
5%           0.015351
25%          0.053469
50%          0.133040
75%          0.337788
95%          1.107650
99%          2.074172
max          8.360016
Name: SEGMENT_LENGTH_KM, dtype: float64

In [0]:
display(
    roads_silver[
        [
            "OBJECTID",
            "RD_NAME",
            "RD_TYPE",
            "CLASSN",
            "RMACLASS",
            "LOCALITY",
            "SEGMENT_LENGTH_KM"
        ]
    ]
    .sort_values(
        "SEGMENT_LENGTH_KM",
        ascending=False
    )
    .head(20)
)

OBJECTID,RD_NAME,RD_TYPE,CLASSN,RMACLASS,LOCALITY,SEGMENT_LENGTH_KM
29473,OMEO,HIGHWAY,HW,AO,MITTA MITTA,8.360015801516338
10931,CALDER,HIGHWAY,HW,AH,HATTAH,7.361844912734037
88225,BACCHUS MARSH-WERRIBEE,ROAD,MR,AO,QUANDONG,7.287818210402315
28804,BENAMBRA-CORRYONG,ROAD,FR,AO,NARIEL VALLEY,7.17947438112056
10896,CALDER,HIGHWAY,HW,AH,HATTAH,7.040672191673556
28872,OMEO,HIGHWAY,HW,AO,MITTA MITTA,6.81091828678022
1154,PRINCES,HIGHWAY EAST,HW,AH,WINGAN RIVER,6.760835988735242
22059,MURRAY RIVER,ROAD,MR,AO,THOLOGOLONG,6.694744942614432
54636,WARBURTON-WOODS POINT,ROAD,MR,AO,CAMBARVILLE,6.513878267722058
27931,DARTMOUTH,ROAD,TR,AO,MITTA MITTA,6.475867722618275


## Spatial Data Quality Flags

Add explicit spatial-quality indicators without removing valid road records.

In [0]:
roads_silver["DQ_ZERO_LENGTH"] = (
    roads_silver["SEGMENT_LENGTH_M"] <= 0
)

roads_silver["DQ_INVALID_GEOMETRY"] = (
    ~roads_silver.geometry.is_valid
)

print(
    "Zero-length segments:",
    roads_silver["DQ_ZERO_LENGTH"].sum()
)

print(
    "Invalid geometries:",
    roads_silver["DQ_INVALID_GEOMETRY"].sum()
)

Zero-length segments: 0
Invalid geometries: 0


In [0]:
roads_silver["DQ_ANY_ISSUE"] = (
    roads_silver[
        [
            "DQ_CORE_ATTRIBUTE_ISSUE",
            "DQ_ZERO_LENGTH",
            "DQ_INVALID_GEOMETRY"
        ]
    ]
    .any(axis=1)
)

roads_silver["DQ_OVERALL_STATUS"] = np.where(
    roads_silver["DQ_ANY_ISSUE"],
    "REVIEW",
    "COMPLETE"
)

print("Overall data-quality status created")

Overall data-quality status created


## Silver Quality Summary

Summarise the key integrity and quality outcomes produced by the Silver layer.

In [0]:
silver_quality_summary = pd.DataFrame({
    "quality_metric": [
        "Total road segments",
        "Unique OBJECTIDs",
        "Duplicate OBJECTIDs",
        "Missing geometries",
        "Invalid geometries",
        "Core attribute issues",
        "Zero-length segments",
        "Records requiring review"
    ],
    "value": [
        len(roads_silver),

        roads_silver[
            "OBJECTID"
        ].nunique(),

        roads_silver[
            "OBJECTID"
        ].duplicated().sum(),

        roads_silver.geometry
        .isna()
        .sum(),

        (
            ~roads_silver.geometry.is_valid
        ).sum(),

        roads_silver[
            "DQ_CORE_ATTRIBUTE_ISSUE"
        ].sum(),

        roads_silver[
            "DQ_ZERO_LENGTH"
        ].sum(),

        roads_silver[
            "DQ_ANY_ISSUE"
        ].sum()
    ]
})

display(silver_quality_summary)

quality_metric,value
Total road segments,90797
Unique OBJECTIDs,90797
Duplicate OBJECTIDs,0
Missing geometries,0
Invalid geometries,0
Core attribute issues,1763
Zero-length segments,0
Records requiring review,1763


## Reporting-Ready Silver View

Create a focused analytical view containing fields required for downstream
Asset Intelligence aggregation.

The complete Silver dataset is retained separately, including optional source
attributes.

In [0]:
reporting_cols = [
    "OBJECTID",

    "DEC_NAME",
    "DEC_TYPE",

    "RD_NAME",
    "RD_TYPE",

    "LOCAL_NAME",
    "LOCAL_TYPE",

    "RD_NUM",
    "RD_SECTION",

    "CLASSN",
    "PROFILE",
    "SRNS",

    "RMANUM",
    "RMACLASS",

    "LOCALITY",

    "SEGMENT_LENGTH_M",
    "SEGMENT_LENGTH_KM",

    "DQ_MISSING_DEC_TYPE",
    "DQ_MISSING_RD_TYPE",
    "DQ_MISSING_LOCAL_TYPE",
    "DQ_CORE_ATTRIBUTE_ISSUE",
    "DQ_ATTRIBUTE_STATUS",

    "DQ_ZERO_LENGTH",
    "DQ_INVALID_GEOMETRY",

    "DQ_ANY_ISSUE",
    "DQ_OVERALL_STATUS",

    "geometry"
]

roads_silver_reporting = (
    roads_silver[
        reporting_cols
    ]
    .copy()
)

print(
    f"Reporting records : {len(roads_silver_reporting):,}"
)

print(
    f"Reporting columns : {len(roads_silver_reporting.columns)}"
)

display(
    roads_silver_reporting.head(10)
)

Reporting records : 90,797
Reporting columns : 27


,OBJECTID,DEC_NAME,DEC_TYPE,RD_NAME,RD_TYPE,LOCAL_NAME,LOCAL_TYPE,RD_NUM,RD_SECTION,CLASSN,PROFILE,SRNS,RMANUM,RMACLASS,LOCALITY,SEGMENT_LENGTH_M,SEGMENT_LENGTH_KM,DQ_MISSING_DEC_TYPE,DQ_MISSING_RD_TYPE,DQ_MISSING_LOCAL_TYPE,DQ_CORE_ATTRIBUTE_ISSUE,DQ_ATTRIBUTE_STATUS,DQ_ZERO_LENGTH,DQ_INVALID_GEOMETRY,DQ_ANY_ISSUE,DQ_OVERALL_STATUS,geometry
0,1,MACKENZIE,ROAD,MACKENZIE,ROAD,SIMS,STREET,5248,01,MR,1,N,5248,AO,WEST MELBOURNE,34.740412,0.034740,False,False,False,False,COMPLETE,False,False,False,COMPLETE,"LINESTRING (144.90811 -37.80662, 144.90826 -37..."
1,2,MACKENZIE,ROAD,MACKENZIE,ROAD,SIMS,STREET,5248,01,MR,8,N,5248,AO,WEST MELBOURNE,44.767868,0.044768,False,False,False,False,COMPLETE,False,False,False,COMPLETE,"LINESTRING (144.90912 -37.80679, 144.90909 -37..."
2,3,MACKENZIE,ROAD,MACKENZIE,ROAD,SIMS,STREET,5248,01,MR,8,N,5248,AO,WEST MELBOURNE,52.743874,0.052744,False,False,False,False,COMPLETE,False,False,False,COMPLETE,"LINESTRING (144.90967 -37.80691, 144.90966 -37..."
3,4,MACKENZIE,ROAD,MACKENZIE,ROAD,SIMS,STREET,5248,01,MR,8,N,5248,AO,WEST MELBOURNE,19.106246,0.019106,False,False,False,False,COMPLETE,False,False,False,COMPLETE,"LINESTRING (144.90909 -37.80696, 144.90912 -37..."
4,5,MACKENZIE,ROAD,MACKENZIE,ROAD,SIMS,STREET,5248,01,MR,8,N,5248,AO,WEST MELBOURNE,10.693296,0.010693,False,False,False,False,COMPLETE,False,False,False,COMPLETE,"LINESTRING (144.90909 -37.80696, 144.909 -37.8..."
5,6,MACKENZIE,ROAD,MACKENZIE,ROAD,SIMS,STREET,5248,01,MR,8,N,5248,AO,WEST MELBOURNE,81.838746,0.081839,False,False,False,False,COMPLETE,False,False,False,COMPLETE,"LINESTRING (144.91058 -37.80704, 144.91013 -37..."
6,7,MACKENZIE,ROAD,MACKENZIE,ROAD,SIMS,STREET,5248,01,MR,1,N,5248,AO,WEST MELBOURNE,55.555896,0.055556,False,False,False,False,COMPLETE,False,False,False,COMPLETE,"LINESTRING (144.90789 -37.80709, 144.90811 -37..."
7,8,MACKENZIE,ROAD,MACKENZIE,ROAD,SIMS,STREET,5248,01,MR,8,N,5248,AO,WEST MELBOURNE,48.238201,0.048238,False,False,False,False,COMPLETE,False,False,False,COMPLETE,"LINESTRING (144.90842 -37.8072, 144.90832 -37...."
8,9,MACKENZIE,ROAD,MACKENZIE,ROAD,SIMS,STREET,5248,01,MR,8,N,5248,AO,WEST MELBOURNE,58.304012,0.058304,False,False,False,False,COMPLETE,False,False,False,COMPLETE,"LINESTRING (144.909 -37.80703, 144.9089 -37.80..."
9,10,MACKENZIE,ROAD,MACKENZIE,ROAD,MACKENZIE,ROAD,5248,01,MR,1,N,5248,AO,WEST MELBOURNE,34.039085,0.034039,False,False,False,False,COMPLETE,False,False,False,COMPLETE,"LINESTRING (144.90777 -37.80738, 144.90789 -37..."


## Persist Silver Dataset

Persist the complete cleaned and enriched Silver road network for downstream
Gold aggregation.

The Silver output preserves the source geometry while adding standardised
attributes, derived spatial measures and quality indicators.

In [0]:
silver_path = "/Volumes/dtp_data/dtp_schema/dtp_geojson/managed_roads_silver.geojson"

roads_silver.to_file(
    silver_path,
    driver="GeoJSON"
)

print("Silver saved successfully")
print("Silver path:")
print(silver_path)

Silver saved successfully
Silver path:
/Volumes/dtp_data/dtp_schema/dtp_geojson/managed_roads_silver.geojson


In [0]:
silver_check = gpd.read_file(
    silver_path
)

print("=== SILVER OUTPUT VERIFICATION ===")

print(
    f"Records : {len(silver_check):,}"
)

print(
    f"Columns : {len(silver_check.columns)}"
)

print(
    f"CRS     : {silver_check.crs}"
)

assert len(silver_check) == len(roads_bronze), \
    "Silver record count changed during persistence"

assert silver_check["OBJECTID"].isna().sum() == 0, \
    "Missing OBJECTIDs detected in persisted Silver"

assert silver_check["OBJECTID"].duplicated().sum() == 0, \
    "Duplicate OBJECTIDs detected in persisted Silver"

assert silver_check.geometry.isna().sum() == 0, \
    "Missing geometries detected in persisted Silver"

assert "SEGMENT_LENGTH_KM" in silver_check.columns, \
    "Derived segment length is missing"

assert "DQ_OVERALL_STATUS" in silver_check.columns, \
    "Overall data-quality status is missing"

print("\nSilver persistence validation: PASS")

=== SILVER OUTPUT VERIFICATION ===
Records : 90,797
Columns : 35
CRS     : EPSG:4326

Silver persistence validation: PASS


## Silver Layer Complete

The validated Bronze DTP Managed Roads dataset has been transformed into
an analysis-ready Silver road-segment dataset.

### Transformations Completed

- Blank and whitespace-only source values were standardised to NULL.
- Legitimate optional missingness was preserved without artificial imputation.
- Incomplete core road-type attributes were identified and flagged.
- Selected categorical fields were standardised.
- Record count and OBJECTID integrity were preserved.
- Geometry integrity was revalidated.
- A projected working geometry was used for distance measurement.
- Segment length was derived in metres and kilometres.
- Attribute and spatial data-quality indicators were created.
- A focused reporting-ready Silver view was prepared.
- The complete Silver dataset was persisted and validated.

The Silver layer is ready for Gold-level Asset Intelligence aggregation
and Power BI reporting.